# 🎯 06 · 数据分析面试八股与高频题

> 最后冲刺：**60 连问速答 + SQL 手撕 + pandas 手撕 + 业务案例分析**。目标：不看书能把口径、方法、坑讲清楚。

## 1️⃣ 指标口径 15 连问（速答）

1. **DAU/MAU**：日/月活跃用户；DAU/MAU 常用来衡量粘性
2. **留存率**：某期新增/活跃用户在后续第 N 天仍活跃的比例
3. **转化率**：完成目标行为人数 / 到达入口人数
4. **ARPU**：总收入 / 活跃用户；**ARPPU**：总收入 / 付费用户
5. **GMV**：成交总额 = 流量 × 转化率 × 客单价
6. **复购率**：多次购买用户占比或购买频次≥2 占比
7. **LTV**：用户生命周期价值；**CAC**：获客成本；LTV/CAC>3 常见健康线
8. **漏斗**：分层转化路径，定位流失环节
9. **同期群**：按注册/首购日分组，避免结构变化干扰
10. **北极星指标**：唯一关键增长指标，与商业价值强相关
11. **A/B 实验**：随机分流、AA/SRM 校验、显著性、功效、peeking 陷阱
12. **辛普森悖论**：分组结论与整体结论相反，需看混杂变量
13. **幸存者偏差**：只看到活下来的样本
14. **同比/环比**：同比=与上年同期比；环比=与上一期比
15. **埋点**：前端/后端数据采集，决定口径一致性

## 2️⃣ 统计学 15 连问（速答）

1. **均值 vs 中位数**：均值受离群值影响大；右偏时均值>中位数
2. **方差/标准差**：衡量离散程度；n 与 n-1 的区别（总体/样本）
3. **IQR 离群规则**：Q1-1.5×IQR / Q3+1.5×IQR 之外
4. **z-score**：(x-μ)/σ，|z|>2 或 3 常视为离群
5. **CLT**：大样本下样本均值近似正态
6. **置信区间**：95% CI 的含义是反复抽样 95% 覆盖真值
7. **p 值**：假设 H0 成立时观测到如此极端结果的概率，不是 H0 为真的概率
8. **一类错误 α / 二类错误 β**：假阳性/假阴性；功效=1-β
9. **t 检验 vs z 检验**：总体方差未知且小样本用 t
10. **卡方检验**：类别变量独立性
11. **相关 vs 因果**：相关≠因果，需实验/因果推断
12. **皮尔逊 r**：线性相关强度，对离群值敏感；Spearman 更稳健
13. **多重比较问题**：比较越多，假阳性越多；Bonferroni/BH-FDR
14. **功效与样本量**：n 随 α↓、β↓、效应↓、σ↑ 而增大
15. **正态性检验**：Shapiro/K-S；非正态可用非参检验

## 3️⃣ SQL 手撕：建表与环境

用 Python 内置 sqlite3 模拟真实 SQL：

In [1]:
import sqlite3
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.execute('CREATE TABLE orders (order_id INT, user_id INT, city TEXT, amount REAL, dt TEXT)')
cur.execute('CREATE TABLE users (user_id INT, reg_dt TEXT)')
rows = [
    (1, 101, '北京', 120.5, '2025-01-05'), (2, 102, '上海', 88.0, '2025-01-06'),
    (3, 103, '北京', 210.5, '2025-01-06'), (4, 101, '北京', 99.0, '2025-01-07'),
    (5, 104, '深圳', 155.0, '2025-01-07'), (6, 102, '上海', 300.0, '2025-01-08'),
    (7, 105, '杭州', 45.0, '2025-01-08'), (8, 103, '北京', 133.5, '2025-01-09'),
]
cur.executemany('INSERT INTO orders VALUES (?,?,?,?,?)', rows)
cur.executemany('INSERT INTO users VALUES (?,?)', [(101,'2025-01-01'),(102,'2025-01-02'),(103,'2025-01-02'),(104,'2025-01-03'),(105,'2025-01-03')])
conn.commit()
print('SQLite 环境就绪')

SQLite 环境就绪


## 4️⃣ SQL 手撕：按城市聚合 + top-N 用户

In [2]:
# 每城市订单量与销售额
q1 = '''SELECT city, COUNT(*) AS cnt, SUM(amount) AS total
       FROM orders GROUP BY city ORDER BY total DESC'''
print(cur.execute(q1).fetchall())

# 每个用户的最大订单金额（分组 top-1）
q2 = '''SELECT user_id, MAX(amount) AS max_amt FROM orders GROUP BY user_id ORDER BY max_amt DESC LIMIT 3'''
print('\n用户最大单:', cur.execute(q2).fetchall())

# 复购用户数：下单次数>=2 的用户
q3 = '''SELECT COUNT(*) FROM (SELECT user_id FROM orders GROUP BY user_id HAVING COUNT(*) >= 2)'''
print('复购用户数:', cur.execute(q3).fetchone()[0])
assert cur.execute(q3).fetchone()[0] == 3
print('✅ SQL 查询正确')

[('北京', 4, 563.5), ('上海', 2, 388.0), ('深圳', 1, 155.0), ('杭州', 1, 45.0)]

用户最大单: [(102, 300.0), (103, 210.5), (104, 155.0)]
复购用户数: 3
✅ SQL 查询正确


## 5️⃣ pandas 手撕：取数三连

In [3]:
# 复现 SQL 逻辑
import pandas as pd
ords = pd.DataFrame(rows, columns=['order_id', 'user_id', 'city', 'amount', 'dt'])

# 1) 按城市聚合
city_agg = ords.groupby('city').agg(cnt=('order_id', 'count'), total=('amount', 'sum')).sort_values('total', ascending=False)
print('城市聚合:')
print(city_agg)

# 2) 复购用户（101/102/103 各下 2 单 → 3 个复购用户）
repeat = ords.groupby('user_id').size()
print('\n复购用户数:', (repeat >= 2).sum())
assert (repeat >= 2).sum() == 3

# 3) 每日销售额
daily = ords.groupby('dt')['amount'].sum()
print('\n每日销售额:')
print(daily.round(2))
print('✅ pandas 与 SQL 结论一致')

城市聚合:
      cnt  total
city            
北京      4  563.5
上海      2  388.0
深圳      1  155.0
杭州      1   45.0

复购用户数: 3

每日销售额:
dt
2025-01-05    120.5
2025-01-06    298.5
2025-01-07    254.0
2025-01-08    345.0
2025-01-09    133.5
Name: amount, dtype: float64
✅ pandas 与 SQL 结论一致


## 6️⃣ 业务案例分析：日活下降

**问题**：某 App 近一周 DAU 下滑 8%，怎么定位？

**分析框架（分层拆解）**：

1. **拆指标**：DAU = 新增 + 回流 - 流失；看三部分各自变化
2. **拆时间**：工作日/周末、环比/同比，排除周期性
3. **拆渠道**：自然量/广告/活动各来源贡献变化
4. **拆版本/功能**：新版本上线后留存是否下滑
5. **外部因素**：竞品、节假日、政策、舆情

> 面试踩分点：先问口径 → 再分层拆解 → 给假设 → 用数据验证 → 给结论与下一步。

In [4]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
rng = np.random.default_rng(9)

# 模拟 DAU 拆解：新增 / 回流 / 留存
new = rng.integers(800, 1000, 7)
back = rng.integers(1500, 1700, 7)
stay = rng.integers(8500, 9300, 7)  # 老用户留存贡献

dau = new + back + stay
print('DAU 组成:')
print(pd.DataFrame({'新增': new, '回流': back, '留存': stay, 'DAU': dau}).T)

plt.figure(figsize=(8, 4))
plt.stackplot(range(7), new, back, stay, labels=['新增', '回流', '留存'], colors=['#60a5fa', '#f59e0b', '#34d399'])
plt.xlabel('第 N 天')
plt.ylabel('用户数')
plt.title('DAU 分层拆解（面积堆叠）')
plt.legend(loc='upper right')
plt.show()
print('\n→ 若「留存」贡献持续下降，问题在新用户质量或产品体验')

DAU 组成:
         0      1      2      3      4      5      6
新增     884    974    992    857    822    920    933
回流    1655   1628   1643   1683   1683   1685   1672
留存    9076   9234   8510   8521   9138   8849   9098
DAU  11615  11836  11145  11061  11643  11454  11703

→ 若「留存」贡献持续下降，问题在新用户质量或产品体验


C:\Users\24260\AppData\Local\Temp\ipykernel_24164\3262269817.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7️⃣ 高频业务题 15 连问（速答）

1. **日活跌了怎么排查**：口径→拆解→渠道/版本/外部
2. **怎么评估新功能**：A/B 实验 + 留存/转化/满意度
3. **怎么定留存目标**：对标品类均值 + 同期群看趋势
4. **怎么衡量活动效果**：增量 vs 自然基线（对照组）
5. **什么是辛普森悖论**：分层结论与整体相反
6. **怎么做用户分层**：RFM/行为/价值分层，分层后差异化运营
7. **复购率怎么提升**：会员、优惠券、推荐、售后体验
8. **客单价怎么提升**：捆绑、满减、升级推荐
9. **怎么验证数据质量**：口径核对、埋点校验、空值/重复率
10. **指标体系怎么建**：北极星 → 漏斗 → 分层 → 监控看板
11. **AB 实验样本量**：与 α、β、MDE、σ 的关系
12. **什么叫 p-hacking**：反复试直到显著
13. **留存和活跃什么关系**：留存是活跃的驱动源
14. **怎么和业务方沟通**：结论先行、给出建议、讲清口径
15. **数据分析师最重要的能力**：把业务问题翻译成数据问题

## 📝 自测清单

- [ ] 指标 15 连问能不看资料讲出口径
- [ ] 统计 15 连问能写出公式与易错点
- [ ] SQL 手撕（聚合/top-N/复购）能现场完成
- [ ] pandas 手撕（groupby/筛选/断言）能现场完成
- [ ] 「日活下降」案例能按框架给出完整排查思路